# Paper §6.1 — Calibrated subspace projection: method + calibration

Defines the calibrated subspace projection mitigation and reproduces the (a − m) calibration on PlotQA + InfoVQA pooled (n ≈ 5,000 wrong-base) for OneVision Main. The K=8 SVD top-direction subspace produced here is the upstream artifact that §6.2 (chosen cell application) and §5.2 (K-subspace sweep) both read.

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` — §6.1 (Method + calibration recipe).

This notebook drives heavy inference stages by `subprocess`-invoking the
existing drivers in `scripts/`. The `RUN_INFERENCE = False` toggle below
lets a reviewer read the full pipeline without GPU access — canonical
CSVs are read straight from disk and figures get re-rendered from them.


## 1 · Setup — paths + subprocess helper

In [ ]:
from __future__ import annotations
import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_main_worktree() -> Path:
    common = subprocess.check_output(
        ["git", "rev-parse", "--git-common-dir"], cwd=Path.cwd(), text=True
    ).strip()
    return Path(common).resolve().parent


def find_worktree_root() -> Path:
    return Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=Path.cwd(), text=True
    ).strip()).resolve()


MAIN     = find_main_worktree()
WORKTREE = find_worktree_root()

# Scripts + configs from the active branch (worktree); gitignored artifacts
# (inputs/, outputs/, docs/insights/_data/) from MAIN.
SCRIPTS    = WORKTREE / "scripts"
CONFIGS    = WORKTREE / "configs"
DATA_DIR   = MAIN / "docs" / "insights" / "_data"
FIGURES    = WORKTREE / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

GPUS = os.environ.get("VLM_ANCHOR_GPUS", "0,1,2,3,4")
RUN_INFERENCE = False  # set True to invoke the heavy driver(s)

print(f"MAIN     = {MAIN}")
print(f"WORKTREE = {WORKTREE}")
print(f"GPUS     = {GPUS}")
print(f"RUN_INFERENCE = {RUN_INFERENCE}")


In [ ]:
def run_cmd(cmd: list[str] | str, *, dry: bool = False, env: dict | None = None) -> int:
    printable = " ".join(shlex.quote(c) for c in cmd) if isinstance(cmd, list) else cmd
    print(f"$ {printable}")
    if dry:
        print("  (dry — RUN_INFERENCE=False)")
        return 0
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    return subprocess.run(cmd, cwd=MAIN, env=full_env,
                          shell=isinstance(cmd, str)).returncode


def save_figure(fig, stem: str, png_dir: Path = FIGURES,
                pdf_dir: Path | None = None):
    png = png_dir / f"{stem}.png"
    fig.savefig(png, bbox_inches="tight", dpi=160)
    print(f"wrote {png}")
    if pdf_dir is not None:
        pdf_dir.mkdir(parents=True, exist_ok=True)
        pdf = pdf_dir / f"{stem}.pdf"
        fig.savefig(pdf, bbox_inches="tight")
        print(f"wrote {pdf}")


## 2 · Algorithm

**Calibrated subspace projection (E6).** Given an open-weight VLM, a
target intervention layer `L` (mid-to-late residual), and a calibration
pool of `(a, m)` paired-stimuli sids (anchored + masked-anchor pairs)
where the model is base-wrong, compute:

```
# --- Calibration (one-time, off-line) ---
for sid in calibration_pool:
    h_a = vlm.residual(layer=L, input=a_sid)        # anchored arm
    h_m = vlm.residual(layer=L, input=m_sid)        # digit-masked arm
    d_sid = h_a - h_m
D = stack(d_sid)                                    # (n, d_model)
U, S, V^T = SVD(D, full_matrices=False)
V_K = V_:K                                          # (K, d_model) basis

# --- Inference (any input, no anchor label needed) ---
def forward_with_mitigation(x):
    install_pre_hook(layer=L+1):
        h := residual at layer L+1 input
        proj = (h @ V_K.T) @ V_K                    # (B, T, d_model)
        h' = h − α * proj                           # remove K directions
        return h'
    return vlm.forward(x)
```

§6 chosen cell: **`L=26, K=8, α=1.0`** on OneVision Main
(`llava-onevision-qwen2-7b-ov`, 28-layer Qwen2 backbone — L=26 = 93 % of
the stack, late residual). Design choice anchored by §5.1 layer probes
(per-model peak heterogeneity around late residual) + §5.2 K-subspace
sweep (K=8 dominates K=1 by ~10× on PlotQA Δdf).


## 3 · Calibration — pooled (a − m) SVD on PlotQA + InfoVQA

`scripts/run_calibrate_subspace_sharded.py` shards eligible sids
round-robin across K GPUs, collects per-shard `(D_wrong, D_all)` tensors,
concatenates, then runs SVD once on the merged matrix.

Outline §6.1 calls for one calibration over the PlotQA + InfoVQA pool;
in practice we calibrate per dataset then merge. The pooled basis tensor
ships in the canonical legacy E6 root at:

```
outputs/e6_steering/<model>/_subspace/subspace_plotqa_infovqa_pooled_n5k_K16.pt
```

The K16 suffix means the SVD retained 16 singular vectors — any K ≤ 16
can be sliced at use time (`V_K = V_all[L, :K, :]`).


In [ ]:
ONEVISION = "llava-onevision-qwen2-7b-ov"
SUBSPACE_PATH = MAIN / "outputs" / "e6_steering" / ONEVISION / "_subspace" / "subspace_plotqa_infovqa_pooled_n5k_K16.pt"
CALIB_DIR_PLOTQA  = MAIN / "outputs" / "e6_steering" / ONEVISION / "calibration_plotqa"
CALIB_DIR_INFOVQA = MAIN / "outputs" / "e6_steering" / ONEVISION / "calibration_infographicvqa"
CALIB_DIR_POOLED  = MAIN / "outputs" / "e6_steering" / ONEVISION / "calibration_plotqa_infovqa_pooled"


def calibrate(dataset_tag: str, config_slug: str, predictions_path: str):
    out_dir = MAIN / "outputs" / "e6_steering" / ONEVISION / f"calibration_{dataset_tag}"
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "run_calibrate_subspace_sharded.py"),
        "--config", str(CONFIGS / f"{config_slug}.yaml"),
        "--model", ONEVISION,
        "--hf-model", "llava-hf/llava-onevision-qwen2-7b-ov-hf",
        "--predictions-path", str(MAIN / predictions_path),
        "--dataset-tag", dataset_tag,
        "--max-calibrate-pairs", "2500",
        "--gpus", GPUS,
        "--out-dir", str(out_dir),
    ]
    return run_cmd(cmd, dry=not RUN_INFERENCE)


# Per-dataset calibrations
calibrate("plotqa", "experiment_e7_plotqa_full",
          "outputs/experiment_e7_plotqa_full/llava-onevision-qwen2-7b-ov/20260502-132624/predictions.jsonl")
calibrate("infographicvqa", "experiment_e7_infographicvqa_full",
          "outputs/experiment_e7_infographicvqa_full/llava-onevision-qwen2-7b-ov/20260502-152105/predictions.jsonl")

# Merge per-dataset calibration outputs into the pooled basis. The merge
# script is not in-repo (legacy artifact produced the canonical
# `calibration_plotqa_infovqa_pooled/` and the corresponding subspace .pt);
# below we just point at the existing canonical pool. RUN_INFERENCE=True
# above only re-creates the per-dataset shards; the pooled SVD remains the
# audited artifact that all §6 and §5.2 numbers rest on.
print()
print("=== canonical pooled calibration (read-only) ===")
print(f"  D matrix:   {CALIB_DIR_POOLED / 'D_all.pt'}")
print(f"  SVD basis:  {SUBSPACE_PATH}")
if SUBSPACE_PATH.exists():
    import torch
    V = torch.load(SUBSPACE_PATH, weights_only=True)
    print(f"  shape:      {tuple(V.shape)}   (layers, K_max, d_model)")
    print(f"  L=26 row:   K_max={V.shape[1]}; slicing V[26, :8, :] gives the §6 K=8 basis")


## 4 · Pooled v_meta sidecar — calibration provenance


In [ ]:
v_meta = CALIB_DIR_POOLED / "v_meta.json"
if v_meta.exists():
    meta = json.loads(v_meta.read_text())
    print(f"  pooled calibration provenance:")
    for k in ("dataset_tag", "source_tags", "n_wrong", "n_all",
              "n_wrong_per_source", "n_all_per_source",
              "n_layers", "d_model", "D_wrong_shape", "D_all_shape"):
        if k in meta:
            print(f"    {k}: {meta[k]}")
else:
    print(f"  (sidecar absent: {v_meta})")


## Summary

Pipeline: per-dataset calibration shards (PlotQA + InfoVQA, ~2,500 pairs
each) → pooled D matrix concat → SVD K=16 → seal as
`_subspace/subspace_plotqa_infovqa_pooled_n5k_K16.pt`.

Downstream uses:
- §5.2 K-subspace sweep slices the same basis at K ∈ {1, 2, 4, 8} and at
  multiple layers + alpha for the pilot grid.
- §6.2 chosen-cell paired-bootstrap CI applies V_K=8 at L=26, α=1.0 across
  5 datasets — see `paper_section_6_2_anchoring_reduction.ipynb`.
- §6.3 capability preservation applies the same projection at inference
  on 6 VLMEvalKit benchmarks — see `paper_section_6_3_capability_preservation.ipynb`.
